In [ ]:
# Install necessary libraries
!pip install -q accelerate peft bitsandbytes transformers trl datasets huggingface_hub

import torch
import os
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from trl import SFTTrainer
from huggingface_hub import login

from google.colab import userdata
userdata.get('hf_secret')
login(token=userdata.get('hf_secret'))

In [ ]:
# --- Configuration ---

base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

dataset_name = "Kaeyze/computer-science-synthetic-dataset"

hub_adapter_name = "JnsDev/tinyllama-1.1b-cs-adapter"
local_adapter_dir = "./tinyllama-adapter"


lora_r = 16
lora_alpha = 32        # Alpha scaling (often 2*r)
lora_dropout = 0.05    # Dropout
lora_target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

output_dir = "./results_cs_tutor" # Directory for checkpoints during training
num_train_epochs = 1              # Number of training epochs (1-3 recommended for LoRA)
per_device_train_batch_size = 2   # Batch size (adjust based on Colab GPU memory)
gradient_accumulation_steps = 4   # Accumulate gradients (effective batch size = batch_size * accumulation_steps)
learning_rate = 2e-4              # Learning rate
optim = "paged_adamw_8bit"        # Optimizer for memory efficiency
save_strategy = "epoch"           # Save checkpoints at the end of each epoch ('steps' is also an option)
logging_steps = 25                # Log training progress frequency
fp16 = True                       # Enable mixed precision (required for 4-bit)
bf16 = False                      # Set to True if your GPU supports bf16 (like A100)
max_grad_norm = 0.3               # Gradient clipping threshold
max_steps = 4000                  # Set to -1 for full epochs, or a positive number for a shorter run
warmup_ratio = 0.03               # Warmup steps proportion
lr_scheduler_type = "constant"    # Learning rate scheduler type ("cosine" is another option)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

# Load base model
print(f"Loading base model: {base_model_name}")
model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
# Configure model settings
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model and Tokenizer loaded.")

In [ ]:
model = prepare_model_for_kbit_training(model)

# Create LoRA configuration
peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=lora_target_modules,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
print("PEFT Model prepared:")
model.print_trainable_parameters()

In [ ]:
# Load the dataset
print(f"Loading dataset: {dataset_name}")
dataset = load_dataset(dataset_name, split="train") # Load the training split
print("Dataset example: " + dataset[0]['input'])


# Define the formatting function
def format_cs_prompt(example):
    instruction = example['input']
    response = example['output']
    formatted_text = f"### Instruction:\n{instruction}\n\n### Response:\n{response}"
    return {"text": formatted_text}
# Apply the formatting function
dataset = dataset.map(format_cs_prompt)

print("Dataset formatted.")
print("Example formatted data:")
print(dataset[0]['text'])

In [ ]:
from trl import SFTConfig

# --- 6. Configure Trainer using SFTConfig ---

print("Configuring SFTConfig...")
trainer_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_strategy=save_strategy,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=True,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard",
    push_to_hub=False,

    dataset_text_field="text",      # Field in the dataset containing the formatted text
    max_seq_length=512,             # Maximum sequence length
    packing=False,                  # Whether to pack sequences
)

# Initialize the SFTTrainer
print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,                    # The PEFT-wrapped model
    args=trainer_config,            # Pass the SFTConfig object
    train_dataset=dataset,          # The formatted training dataset
    peft_config=peft_config,        # The PEFT configuration

)

print("Trainer initialized.")

In [ ]:
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning finished!")

In [ ]:
print(f"Pushing adapter to Hub: {hub_adapter_name}")
# Push the trained adapter weights and tokenizer config
trainer.model.push_to_hub(hub_adapter_name, commit_message="Fine-tuned TinyLlama CS Tutor Adapter")
tokenizer.push_to_hub(hub_adapter_name, commit_message="Tokenizer for TinyLlama CS Tutor Adapter")
print("Adapter pushed to Hub successfully!")